# 🏰 Citadel Backend Onboarding - Testing Center

## Explore How to Bring New AI Backends into the Citadel Governance Hub!

Use this Jupyter notebook with Python code snippets to onboard all your AI backends and all associated routing logic, including:
- Configuring LLM backend endpoints and authentication
- Generating Bicep parameter files for deployment
- Deploying backend pools and policy fragments
- Testing deployed models through multiple API formats
- Verifying streaming and SDK integration

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub. If you haven't done so, please refer to the [Citadel Governance Hub Deployment Guide](../guides/full-deployment-guide.md) or [Citadel Governance Hub Quick Deployment Guide](../guides/quick-deployment-guide.md) before proceeding.

## Azure Prerequisites

- An existing Citadel Governance Hub deployment with APIM configured
- LLM backend endpoints (AI Foundry, Azure OpenAI, or external) provisioned and accessible
- Managed Identity configured for APIM authentication to backend services

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Configure the following variables according to your environment before running the notebook:

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

inference_api_version = "2024-05-01-preview"

# ============================================================================
# REQUIRED: Update these values for your environment
# ============================================================================
governance_hub_resource_group = "REPLACE"  ## specify the resource group name where the Governance Hub is located
location = "REPLACE"  ## e.g., "eastus", "westus2", etc.

# ============================================================================
# OPTIONAL: LLM Backend Configuration (pre-configured with sample values from main.bicepparam)
# These values match the AI Foundry backends defined in the template
#
# Required Properties:
# - backendId: Unique identifier (used in APIM backend resource name)
# - backendType: 'ai-foundry' | 'azure-openai' | 'external'
# - endpoint: Base URL of the LLM service
# - authScheme: 'managedIdentity' | 'apiKey' | 'token'
# - supportedModels: Array of model objects with per-model metadata
#
# Model Object Properties:
# - name: Model name (required)
# - sku: SKU name for the deployment (default: 'Standard')
# - capacity: Capacity/TPM quota (default: 100)
# - modelFormat: Model format identifier, e.g., 'OpenAI', 'DeepSeek', 'Microsoft' (default: 'OpenAI')
# - modelVersion: Version of the model (default: '1')
# - retirementDate: Retirement date in YYYY-MM-DD format (optional)
# - apiVersion: API version for OpenAI-type requests (default: '2024-02-15-preview')
# - timeout: Request timeout in seconds (default: 120)
# - inferenceApiVersion: API version for inference-type requests (optional, for non-OpenAI models)
#
# Optional Properties (for load balancing):
# - priority: 1-5, default 1 (lower = higher priority)
# - weight: 1-1000, default 100 (higher = more traffic share)
# ============================================================================

#### WARNING: UPDATE THIS CONFIGURATION BASED ON YOUR DEPLOYED BACKENDS ####
#### If you used azd to deploy backends as well, you can copy the current backend configuration using `azd env get-value LLM_BACKEND_CONFIG` and paste it here, then update the values as needed. ####
llm_backends_config = [
  {
    "authScheme": "managedIdentity",
    "backendId": "aif-REPLACE-0",
    "backendType": "ai-foundry",
    "endpoint": "https://aif-REPLACE-0.cognitiveservices.azure.com/",
    "priority": 1,
    "supportedModels": [
      {
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "2024-07-18",
        "name": "gpt-4o-mini",
        "retirementDate": "2026-09-30",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "2024-11-10",
        "name": "gpt-4o",
        "retirementDate": "2026-09-30",
        "sku": "GlobalStandard"
      },
      {
        "apiVersion": "2025-04-01-preview",
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "2025-04-14",
        "name": "gpt-4.1",
        "retirementDate": "2026-10-14",
        "sku": "GlobalStandard",
        "timeout": 180
      },
      {
        "capacity": 1,
        "inferenceApiVersion": "2024-05-01-preview",
        "modelFormat": "DeepSeek",
        "modelVersion": "1",
        "name": "DeepSeek-R1",
        "retirementDate": "2099-12-30",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 1,
        "inferenceApiVersion": "2024-05-01-preview",
        "modelFormat": "Microsoft",
        "modelVersion": "3",
        "name": "Phi-4",
        "retirementDate": "2099-12-30",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "1",
        "name": "text-embedding-3-large",
        "retirementDate": "2027-04-14",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "2026-03-05",
        "name": "gpt-5.4",
        "retirementDate": "2027-03-17",
        "sku": "GlobalStandard"
      }
    ],
    "weight": 100
  },
  {
    "authScheme": "managedIdentity",
    "backendId": "aif-REPLACE-1",
    "backendType": "ai-foundry",
    "endpoint": "https://aif-REPLACE-1.cognitiveservices.azure.com/",
    "priority": 1,
    "supportedModels": [
      {
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "2025-08-07",
        "name": "gpt-5",
        "retirementDate": "2027-02-05",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 1,
        "inferenceApiVersion": "2024-05-01-preview",
        "modelFormat": "DeepSeek",
        "modelVersion": "1",
        "name": "DeepSeek-R1",
        "retirementDate": "2099-12-30",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 100,
        "modelFormat": "OpenAI",
        "modelVersion": "1",
        "name": "text-embedding-3-large",
        "retirementDate": "2027-04-14",
        "sku": "GlobalStandard"
      },
      {
        "capacity": 150,
        "modelFormat": "OpenAI",
        "modelVersion": "2026-03-05",
        "name": "gpt-5.4",
        "retirementDate": "2027-03-17",
        "sku": "GlobalStandard"
      }
    ],
    "weight": 100
  }
]

# Managed Identity for APIM authentication (will be auto-discovered if not specified)
apim_managed_identity_name = ""  # Leave empty to auto-discover

utils.print_info(f"Initialization completed.")

<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [ ]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    
    apim_resource_name = apimClientTool.apim_resource_name
    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    
    utils.print_ok(f"APIM Client Tool initialized successfully!")
    utils.print_info(f"APIM Resource Name: {apim_resource_name}")
    utils.print_info(f"APIM Gateway URL: {apim_resource_gateway_url}")
    
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

<a id='3'></a>
### 3️⃣ Extract Current APIM Backend-Pools Configuration

Retrieve and analyze the existing backend pools and backends configured in your APIM instance:

In [ ]:
# Extract current backends from APIM using the SDK
utils.print_info("Extracting current APIM backends configuration...")

try:
    # Use the APIMClientTool's new get_backends method (uses Azure SDK instead of CLI)
    existing_backends, existing_backend_pools = apimClientTool.get_backends()
    
except Exception as e:
    utils.print_error(f"Error extracting backends: {e}")
    existing_backends = []
    existing_backend_pools = []

In [ ]:
# Get supported models from the policy fragment (if exists)
try:
    supported_models_from_policy = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Supported models in APIM policy fragment 'set-backend-pools':")
    for model in supported_models_from_policy:
        print(f"  • {model}")
except Exception as e:
    utils.print_warning(f"Could not retrieve policy fragment (may not exist yet): {e}")
    supported_models_from_policy = []

In [ ]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

<a id='4'></a>
### 4️⃣ Discover Managed Identity for APIM Authentication

Auto-discover or specify the user-assigned managed identity used by APIM:

In [ ]:
# Discover managed identity from APIM using the SDK
utils.print_info("Discovering managed identity configuration...")

# Use the APIMClientTool's get_managed_identity_info method
managed_identity_info = apimClientTool.get_managed_identity_info()

managed_identity_client_id = managed_identity_info.get('clientId')
managed_identity_name = managed_identity_info.get('name') or apim_managed_identity_name
managed_identity_resource_group = managed_identity_info.get('resourceGroup') or governance_hub_resource_group

if not managed_identity_client_id:
    utils.print_warning("Could not auto-discover managed identity. Please specify it manually in the configuration.")
else:
    utils.print_info(f"Client ID: {managed_identity_client_id}")

if managed_identity_name:
    utils.print_ok(f"Managed Identity Name: {managed_identity_name}")
    utils.print_ok(f"Managed Identity Resource Group: {managed_identity_resource_group}")

<a id='5'></a>
### 5️⃣ Generate LLM Backend Parameter File

Generate a customizable `.bicepparam` file with the full list of LLM backends to be integrated with APIM:

In [ ]:
# Configure the LLM backends for deployment
# You can modify the llm_backends_config list defined in the initialization cell

utils.print_info("LLM Backends to be deployed:")
for backend in llm_backends_config:
    print(f"\n  🔗 {backend['backendId']}")
    print(f"     Type: {backend['backendType']}")
    print(f"     Endpoint: {backend['endpoint']}")
    print(f"     Auth: {backend['authScheme']}")
    print(f"     Priority: {backend.get('priority', 1)}, Weight: {backend.get('weight', 100)}")
    # Display supported models with their per-model metadata
    print(f"     Models ({len(backend['supportedModels'])}):")
    for model in backend['supportedModels']:
        model_name = model['name'] if isinstance(model, dict) else model
        if isinstance(model, dict):
            sku = model.get('sku', 'Standard')
            capacity = model.get('capacity', 100)
            fmt = model.get('modelFormat', 'OpenAI')
            ver = model.get('modelVersion', '1')
            extras = []
            if 'retirementDate' in model:
                extras.append(f"Retirement: {model['retirementDate']}")
            if 'apiVersion' in model:
                extras.append(f"API: {model['apiVersion']}")
            if 'timeout' in model:
                extras.append(f"Timeout: {model['timeout']}s")
            if 'inferenceApiVersion' in model:
                extras.append(f"Inference API: {model['inferenceApiVersion']}")
            extra_str = f", {', '.join(extras)}" if extras else ""
            print(f"       - {model_name} (SKU: {sku}, Capacity: {capacity}, Format: {fmt}, Version: {ver}{extra_str})")
        else:
            print(f"       - {model_name}")

In [ ]:
# Generate the .bicepparam file content
bicep_dir = "../bicep/infra/llm-backend-onboarding"
params_file = os.path.join(bicep_dir, "llm-backends-generated-local.bicepparam")

# Format a single model object for Bicep
def format_model_for_bicep(model):
    """Format a model object for Bicep with per-model metadata including optional routing attributes."""
    if isinstance(model, str):
        # Legacy format: just model name string - convert to object with defaults
        return f"{{ name: '{model}' }}"
    
    # New format: model object with metadata
    parts = [f"name: '{model['name']}'"]
    if 'sku' in model:
        parts.append(f"sku: '{model['sku']}'")
    if 'capacity' in model:
        parts.append(f"capacity: {model['capacity']}")
    if 'modelFormat' in model:
        parts.append(f"modelFormat: '{model['modelFormat']}'")
    if 'modelVersion' in model:
        parts.append(f"modelVersion: '{model['modelVersion']}'")
    if 'retirementDate' in model:
        parts.append(f"retirementDate: '{model['retirementDate']}'")
    if 'apiVersion' in model:
        parts.append(f"apiVersion: '{model['apiVersion']}'")
    if 'timeout' in model:
        parts.append(f"timeout: {model['timeout']}")
    if 'inferenceApiVersion' in model:
        parts.append(f"inferenceApiVersion: '{model['inferenceApiVersion']}'")
    
    return "{ " + ", ".join(parts) + " }"

# Format backends array for Bicep (uses per-model metadata)
def format_backend_for_bicep(backend):
    """Format a backend configuration for Bicep with per-model metadata."""
    # Format each model with its individual metadata
    models_formatted = [format_model_for_bicep(m) for m in backend['supportedModels']]
    models_str = "\n      ".join(models_formatted)
    
    return f"""  {{
    backendId: '{backend['backendId']}'
    backendType: '{backend['backendType']}'
    endpoint: '{backend['endpoint']}'
    authScheme: '{backend['authScheme']}'
    supportedModels: [
      {models_str}
    ]
    priority: {backend.get('priority', 1)}
    weight: {backend.get('weight', 100)}
  }}"""

backends_bicep_str = "\n".join([format_backend_for_bicep(b) for b in llm_backends_config])

params_content = f"""using './main.bicep'

// ============================================================================
// LLM Backend Onboarding - Generated Parameter File
// Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}
// ============================================================================

// ============================================================================
// API Management (APIM) Configuration
// ============================================================================
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apim_resource_name}'
}}

// ============================================================================
// APIM Managed Identity Configuration
// ============================================================================
param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

// ============================================================================
// LLM Backend Configuration Array
// Each model in supportedModels has its own metadata (sku, capacity, modelFormat, modelVersion)
// Optional routing attributes: retirementDate, apiVersion, timeout, inferenceApiVersion
// ============================================================================
param llmBackendConfig = [
{backends_bicep_str}
]

// ============================================================================
// Circuit Breaker Configuration
// ============================================================================
param configureCircuitBreaker = true
"""

# Write the parameter file
utils.print_info(f"Generating parameter file: {params_file}")
with open(params_file, 'w') as f:
    f.write(params_content)

utils.print_ok(f"Parameter file generated successfully!")
print("\n" + "="*60)
print("GENERATED PARAMETER FILE CONTENT:")
print("="*60)
print(params_content)

<a id='6'></a>
### 6️⃣ Deploy LLM Backend Onboarding Bicep

Deploy the LLM backends, backend pools, and policy fragments to APIM:

In [ ]:
# Deploy the LLM backend onboarding
deployment_name = f"llm-backend-onboarding-{time.strftime('%Y%m%d%H%M%S')}"
template_file = os.path.join(bicep_dir, "main.bicep")

utils.print_info(f"Starting deployment: {deployment_name}")
utils.print_info(f"Template: {template_file}")
utils.print_info(f"Parameters: {params_file}")

# Run the subscription-level deployment
deployment_cmd = f"az deployment sub create --name {deployment_name} --location {location} --template-file {template_file} --parameters {params_file}"

output = utils.run(
    deployment_cmd,
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

if output.success:
    utils.print_ok("Deployment completed successfully!")
    
    # Display deployment outputs if available
    outputs = output.json_data.get('properties', {}).get('outputs', {}) if output.json_data else {}
    
    if outputs:
        print("\n" + "="*60)
        print("DEPLOYMENT OUTPUTS:")
        print("="*60)
        
        for key, value in outputs.items():
            print(f"  {key}: {value.get('value')}")
    else:
        utils.print_info("No deployment outputs returned.")
else:
    utils.print_error("Deployment failed. Check the error messages above.")

<a id='7'></a>
### 7️⃣ Verify Deployed Configuration

Verify that the backends, pools, and policy fragments were created successfully:

In [ ]:
# Re-initialize APIM client to pick up new backends
apimClientTool.initialize()

# Get updated supported models from policy fragment
try:
    updated_supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_ok(f"Updated supported models in APIM policy fragment 'set-backend-pools':")
    for model in updated_supported_models:
        print(f"  • {model}")
except Exception as e:
    utils.print_error(f"Error retrieving policy fragment: {e}")
    updated_supported_models = []

In [ ]:
# Display summary of current configuration
utils.print_info("\n" + "="*60)
utils.print_info("CURRENT APIM BACKEND CONFIGURATION SUMMARY")
utils.print_info("="*60)

if existing_backends:
    print("\n📋 Individual Backends:")
    for backend in existing_backends:
        print(f"  • {backend['name']}")
        print(f"    URL: {backend['url']}")
        if backend['supportedModels']:
            print(f"    Models: {', '.join(backend['supportedModels'])}")

if existing_backend_pools:
    print("\n📦 Backend Pools:")
    for pool in existing_backend_pools:
        print(f"  • {pool['name']}")
        for svc in pool['services']:
            print(f"    - {svc.get('id', 'N/A')} (priority: {svc.get('priority', 'N/A')}, weight: {svc.get('weight', 'N/A')})")

if supported_models_from_policy:
    print(f"\n🤖 Total Supported Models: {len(supported_models_from_policy)}")
    print(f"   {', '.join(supported_models_from_policy)}")

<a id='8'></a>
### 8️⃣ Verify Get Available Models Policy Fragment

Verify that the new `get-available-models` policy fragment was created successfully:

In [ ]:
# Verify the get-available-models policy fragment exists
try:
    # Use the Azure SDK to check if the policy fragment exists
    policy_fragment = apimClientTool.client.policy_fragment.get(
        resource_group_name=apimClientTool.resource_group_name,
        service_name=apimClientTool.apim_resource_name,
        id="get-available-models"
    )
    
    if policy_fragment:
        utils.print_ok("Policy fragment 'get-available-models' exists!")
        utils.print_info("This fragment returns available model deployments in a format similar to Azure Cognitive Services API.")
        utils.print_info(f"Description: {policy_fragment.description}")
except Exception as e:
    utils.print_warning(f"Could not retrieve 'get-available-models' policy fragment: {e}")
    utils.print_info("This fragment will be created after running the deployment.")

<a id='test-foundry'></a>
### 🧪 Test GET /deployments (Microsoft Foundry Integration)

Test the `GET /deployments` endpoint which leverages the `get-available-models` policy fragment. This endpoint is used by Microsoft Foundry to discover available model deployments:

In [ ]:
# Test GET /deployments endpoint (Microsoft Foundry integration)
# This endpoint uses the get-available-models policy fragment to return available model deployments
# Testing both Universal LLM API and Azure OpenAI API endpoints

def test_get_deployments(base_endpoint, api_name, api_key):
    """Test GET /deployments for a given API endpoint."""
    deployments_url = f"{base_endpoint}deployments?api-version={inference_api_version}"
    utils.print_info(f"\nTesting {api_name} GET /deployments: {deployments_url}")
    
    try:
        response = requests.get(
            deployments_url,
            headers={"api-key": api_key},
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = response.json()
            deployments = data.get("value", [])
            
            utils.print_ok(f"{api_name} GET /deployments returned {len(deployments)} model deployment(s)")
            
            print("\n" + "="*60)
            print(f"AVAILABLE MODEL DEPLOYMENTS ({api_name})")
            print("="*60)
            
            for deployment in deployments:
                print(f"\n📦 Deployment: {deployment.get('name', 'N/A')}")
                print(f"   ID: {deployment.get('id', 'N/A')}")
                print(f"   Type: {deployment.get('type', 'N/A')}")
                
                sku = deployment.get('sku', {})
                if sku:
                    print(f"   SKU: {sku.get('name', 'N/A')} (Capacity: {sku.get('capacity', 'N/A')})")
                
                props = deployment.get('properties', {})
                if props:
                    model = props.get('model', {})
                    if model:
                        print(f"   Model: {model.get('name', 'N/A')} (Format: {model.get('format', 'N/A')}, Version: {model.get('version', 'N/A')})")
                    
                    capabilities = props.get('capabilities', {})
                    if capabilities:
                        caps_list = [k for k, v in capabilities.items() if v == 'true' or v == True]
                        print(f"   Capabilities: {', '.join(caps_list) if caps_list else 'N/A'}")
                    
                    print(f"   Status: {props.get('provisioningState', 'N/A')}")
            
            print("\n" + "="*60)
            utils.print_ok(f"{api_name} GET /deployments test completed successfully!")
            return True
        else:
            utils.print_error(f"{api_name} GET /deployments failed: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except Exception as e:
        utils.print_error(f"{api_name} GET /deployments test failed: {str(e)}")
        return False

# Get an API key from subscriptions
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[-1].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[-1].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

if api_key:
    # Test 1: Universal LLM API - GET /models/deployments
    apimClientTool.discover_api("models")
    azure_endpoint_models = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Universal LLM API Base Endpoint: {azure_endpoint_models}models")
    test_get_deployments(f"{azure_endpoint_models}models/", "Universal LLM API", api_key)
    
    # Test 2: Azure OpenAI API - GET /openai/deployments
    try:
        apimClientTool.discover_api("openai")
        azure_endpoint_openai = str(apimClientTool.azure_endpoint)
        utils.print_info(f"\nAzure OpenAI API Base Endpoint: {azure_endpoint_openai}openai")
        test_get_deployments(f"{azure_endpoint_openai}openai/", "Azure OpenAI API", api_key)
    except Exception as e:
        utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
else:
    utils.print_warning("Cannot test GET /deployments - missing API key")

---
## 🧪 Test Deployed Models

The following sections test the deployed models through both the Universal LLM API and Azure OpenAI API endpoints.
---

<a id='test-universal'></a>
### 🧪 Test via Universal LLM API (models/chat/completions)

Test the deployed models using the Universal LLM API which routes based on the `model` field in the request body:

In [ ]:
# Discover the Universal LLM API endpoint
apimClientTool.discover_api("models")
azure_endpoint_models = str(apimClientTool.azure_endpoint)
chat_completions_url_models = f"{azure_endpoint_models}models/chat/completions?api-version={inference_api_version}"

utils.print_info(f"Universal LLM API Endpoint: {chat_completions_url_models}")

# Get an API key from subscriptions (it will get the most recently created Access Contract which should be the one we created for testing)
if apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[-1].get("key")
    utils.print_ok(f"Using subscription: {apimClientTool.apim_subscriptions[-1].get('name')}")
else:
    utils.print_error("No APIM subscriptions found. Please create a subscription first.")
    api_key = None

In [ ]:
# Test each supported model via Universal LLM API
if api_key and updated_supported_models:
    utils.print_info(f"\nTesting {len(updated_supported_models)} models via Universal LLM API...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is 2+2? Answer in one word."}
    ]
    
    for model_name in updated_supported_models[:3]:  # Test first 3 models
        utils.print_info(f"Testing model: {model_name}")
        
        payload = {
            "model": model_name,
            "messages": test_messages
        }
        
        try:
            response = requests.post(
                chat_completions_url_models,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run tests - missing API key or supported models")

<a id='test-openai'></a>
### 🧪 Test via Azure OpenAI API (openai/deployments/{model}/chat/completions)

Test the deployed models using the Azure OpenAI compatible API which uses the deployment name in the URL path:

In [ ]:
# Discover the Azure OpenAI API endpoint
try:
    apimClientTool.discover_api("openai")
    azure_endpoint_openai = str(apimClientTool.azure_endpoint)
    utils.print_info(f"Azure OpenAI API Base Endpoint: {azure_endpoint_openai}")
except Exception as e:
    utils.print_warning(f"Azure OpenAI API not found in APIM: {e}")
    azure_endpoint_openai = None

In [ ]:
# Test models via Azure OpenAI API format
if api_key and azure_endpoint_openai and updated_supported_models:
    utils.print_info(f"\nTesting models via Azure OpenAI API format...\n")
    
    test_messages = [
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What is the capital of France? Answer in one word."}
    ]
    
    for model_name in updated_supported_models[:3]:  # Test first 3 models
        utils.print_info(f"Testing model: {model_name}")
        
        # Azure OpenAI format uses deployment name in URL path
        chat_completions_url_openai = f"{azure_endpoint_openai}openai/deployments/{model_name}/chat/completions?api-version={inference_api_version}"
        
        payload = {
            "messages": test_messages  # No model field needed - it's in the URL
        }
        
        try:
            response = requests.post(
                chat_completions_url_openai,
                headers={"api-key": api_key},
                json=payload,
                timeout=60
            )
            
            utils.print_response_code(response)
            
            if response.status_code == 200:
                data = response.json()
                answer = data.get("choices", [{}])[0].get("message", {}).get("content", "No response")
                region = response.headers.get("x-ms-region", "unknown")
                print(f"  💬 Response: {answer}")
                print(f"  📍 Backend Region: {region}")
                utils.print_ok(f"Model '{model_name}' - SUCCESS\n")
            else:
                utils.print_error(f"Model '{model_name}' - FAILED: {response.text}\n")
                
        except Exception as e:
            utils.print_error(f"Model '{model_name}' - ERROR: {str(e)}\n")
else:
    utils.print_warning("Cannot run Azure OpenAI API tests - missing API key, endpoint, or supported models")

<a id='test-sdk'></a>
### 🧪 Test using Azure OpenAI Python SDK

Test using the official Azure OpenAI Python SDK:

In [ ]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict.
    The AzureOpenAI SDK sends both api-key and Authorization: Bearer headers,
    but APIM tries to validate the Authorization header as a JWT token."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing with Azure OpenAI SDK using model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello from Azure OpenAI SDK!'"}
            ]
        )
        
        utils.print_ok("SDK Test Successful!")
        print(f"💬 Response: {response.choices[0].message.content}")
        print(f"📊 Usage: {response.usage.total_tokens} tokens")
        
    except Exception as e:
        utils.print_error(f"SDK Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run SDK test - missing prerequisites")

<a id='test-streaming'></a>
### 🧪 Test Streaming Response

Test streaming responses using the Azure OpenAI SDK:

In [ ]:
from openai import AzureOpenAI
import httpx

def strip_auth_header(request):
    """Remove Authorization header to prevent APIM JWT validation conflict."""
    request.headers.pop("authorization", None)

if api_key and azure_endpoint_openai and updated_supported_models:
    model_name = updated_supported_models[0]  # Use first available model
    utils.print_info(f"Testing streaming with model: {model_name}")
    
    try:
        client = AzureOpenAI(
            azure_endpoint=azure_endpoint_openai,
            api_key=api_key,
            api_version=inference_api_version,
            http_client=httpx.Client(
                event_hooks={"request": [strip_auth_header]}
            )
        )
        
        start_time = time.time()
        
        response = client.chat.completions.with_raw_response.create(
            model=model_name,
            messages=[
                {"role": "user", "content": "Count from 1 to 10 with commas between each number."}
            ],
            stream=True
        )
        
        print(f"📡 x-ms-region: {response.headers.get('x-ms-region', 'unknown')}")
        print(f"📡 x-ms-stream: {response.headers.get('x-ms-stream', 'N/A')}")
        print("\n💬 Streaming response:")
        
        completion = response.parse()
        collected_content = []
        
        for chunk in completion:
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                collected_content.append(content)
                print(content, end='', flush=True)
        
        elapsed = time.time() - start_time
        print(f"\n\n✅ Stream completed in {elapsed:.2f} seconds")
        print(f"📝 Full response: {''.join(collected_content)}")
        
    except Exception as e:
        utils.print_error(f"Streaming Test Failed: {str(e)}")
else:
    utils.print_warning("Cannot run streaming test - missing prerequisites")

---
## 📊 Results Summary
---

This notebook completed the following tasks:

1. **Extracted** current APIM backend-pools configurations
2. **Generated** a customizable LLM backend parameter file (`.bicepparam`)
   - Includes per-model metadata fields: `sku`, `capacity`, `modelFormat`, `modelVersion`
3. **Deployed** the LLM onboarding Bicep templates
4. **Verified** deployment including the new `get-available-models` policy fragment
5. **Tested** the deployed models through:
   - **GET /deployments** (Microsoft Foundry integration - uses `get-available-models` policy fragment)
   - Universal LLM API (`/models/chat/completions`)
   - Azure OpenAI API (`/openai/deployments/{model}/chat/completions`)
   - Azure OpenAI Python SDK
   - Streaming responses

### Next Steps

- Modify the `llm_backends_config` in the initialization cell to add more backends
- Include per-model metadata (`sku`, `capacity`, `modelFormat`, `modelVersion`) for accurate model info
- Re-run the deployment cells to update the APIM configuration
- Use the generated parameter file as a template for CI/CD pipelines
- Use the `get-available-models` policy fragment to expose available models to Microsoft Foundry and other clients

<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

The backends and policy fragments deployed by this notebook are intended to persist as part of your Governance Hub configuration. To remove them, delete the corresponding backend resources and policy fragments from your APIM instance via the Azure portal or CLI.

> **Note:** Removing backends will affect any access contracts that reference those backend pools. Ensure no active contracts depend on them before cleanup.